# MindFlow Step 5 · Summary across the steps

**Question.** What does the whole line of reasoning show, in one figure and one parameter table?


Reads the newest finished run of every step from `MindFlow/runs/` (or the `RUNS` set below) and puts the headline cell
of each step side by side. Nothing is trained here; it runs in a minute on CPU.


<details><summary>Where this notebook sits in the MindFlow</summary>


| step | question | notebooks |
|---|---|---|
| **1** | Does the JKX chart signal exist at the daily frequency on our large caps (also for pretrained models)? | `Step1a_Daily_JKX_CNN`, `Step1b_Daily_Pretrained` |
| **2** | Moved to 5-minute charts: does any model (CNN, ResNet, 1D CNN, ViT, HGB, frozen DINOv2/CLIP, time-series foundation models) find direction? | `Step2a_Intraday_Direction_Models`, `Step2b_Intraday_Direction_Pretrained` |
| **3** | Same models, target = volatility: is there a signal, and who wins? | `Step3a_Intraday_Volatility_Models`, `Step3b_Intraday_Volatility_Pretrained` |
| **4** | Add information to the picture step by step until the image model reaches HGB; is the image still useful beyond the features; which features matter? | `Step4a_Add_Image_Information`, `Step4b_Feature_Selection` |
| **5** | One figure and one parameter table over all steps | `Step5_Summary` |
| **6** | Next questions: does market-regime information (VIX, multi-day realized volatility) help? | `Step6a_VIX_Regime`, `Step6b_Final_Model` (final model + held-out test), `Step6c_Volatility_Use` (IC, long-short, risk sizing), `Step6d_Pure_Image` (the numbers drawn into the picture, no numbers input), `Step6e_Pure_Image_60min` |
| **7** | Efficient computing for the final model: 16-bit / INT8 inference, LoRA fine-tuning of a pretrained ViT | `Step7a_Efficient_Computing`, `Step7b_Distillation` (small students taught by the final CNN), `Step7b_plus_Pure_Image_Distillation` (small pure-image students taught by the 6d pure-image CNN) |

</details>

**Data.** `alldata_bundle_v4.zip` (TAQ filter v4, point-in-time price adjustment): S&P 500 ∪ NASDAQ 100 ∪ DJIA 30
members by date, 2,514 sessions 2016-01-04 … 2025-12-31, five-minute bars plus SPY on the same clock.
**Split.** Train 2016–2022 · validation 2023 (every choice) · test 2024–2025 (last section, run once at the end).
**Advisor rule.** Every model prints its parameter count; for HGB the count is tree nodes.

## 1. Setup
Put `alldata_bundle_v4.zip` and the `code/` folder in `Demo/MindFlow` on Drive. The bundle (1.3 GB) is copied to the
Colab disk and unzipped once per runtime; the next notebook in the same runtime reuses it.

In [ ]:
import os, sys, json, shutil, time
from pathlib import Path

LOCAL_TEST = os.environ.get("MINDFLOW_LOCAL_TEST") == "1"          # repository QA only (synthetic data)
if LOCAL_TEST:
    HERE, LOCAL_ROOT = Path(os.environ["MINDFLOW_HERE"]), Path(os.environ["MINDFLOW_LOCAL"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    HERE, LOCAL_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/EC/Project/Demo/MindFlow"), Path("/content/mindflow")
CODE = HERE / "code"
assert (CODE / "mindflow.py").is_file(), f"Put the code/ folder in {HERE}"
sys.path[:0] = [str(CODE)]
import mindflow as mf
PREPARED = mf.stage_bundle(HERE / mf.BUNDLE_NAME, LOCAL_ROOT)
PANELS_ADDED = mf.stage_panels(HERE, PREPARED)          # e.g. vixy_panel_v4.zip, if uploaded
import numpy as np, pandas as pd, torch
from IPython.display import display, Markdown

DEVICE = torch.device("cpu")

## 2. Which runs, which headline cells

In [ ]:
RUNS = {s: None for s in ("1a", "1b", "2a", "2b", "3a", "3b", "4a", "4b")}   # None = newest run of that step
HEADLINE = {"1": "dir_I20R5", "2": "xsdir_h12", "3": "vol_h1", "4": "vol_h1"}  # one cell shown per step
FOUND = {s: mf.latest_run(HERE, s, r) for s, r in RUNS.items()}
for s, p in FOUND.items():
    print(f"Step {s}: {p if p else '— not run yet'}")
OUT = mf.run_folder(HERE, "5"); OUT.mkdir(parents=True, exist_ok=True)

# Steps 2b / 3b written before frozen_probe 2026-11-24.1 took the foundation models' AUC from scores clipped
# to [0, 1] (ties pulled it towards 0.5; the intervals were right). Rescore them once from the saved forecasts.
import pretrained_ts as pt
for s, ref in (("2b", "2a"), ("3b", "3a")):
    if FOUND[s] is not None and not (FOUND[s] / "tsfm_table_before_2026-11-24.csv").exists():
        pt.rebuild_tsfm_table(FOUND[s], FOUND[ref], n_boot=50 if LOCAL_TEST else 300)

## 3. Headline AUCs per step

In [ ]:
def family(label):
    l = label.lower()
    if "hgb" in l: return "tree"
    if "vit" in l and "probe" not in l: return "ViT"
    if "random walk" in l: return "series"
    if any(k in l for k in ("probe", "chronos", "timesfm", "moirai", "kronos", "zero-shot")): return "pretrained"
    if "1d" in l or "series" in l: return "series"
    if "numbers only" in l or "features only" in l: return "features"
    if "rule" in l or "prior" in l: return None
    return "image"

rows = []
def add(step, frame, label_col, auc, low, high):
    for _, r in frame.iterrows():
        f = family(str(r[label_col]))
        if f: rows.append(dict(step=step, label=str(r[label_col]), family=f, auc=r[auc], low=r[low], high=r[high]))

for s in ("1a", "1b"):
    t = mf.read_csv(FOUND[s] / ("daily_table.csv" if s == "1a" else "daily_pretrained_table.csv")) if FOUND[s] else None
    if t is not None:
        t = t[(t.cell == HEADLINE["1"]) & (t.split == "validation")]
        if s == "1b": t = t[~t.raw.isin(["cnn", "hgb"]) & ~t.raw.str.startswith("rule")]
        add("1", t, "model", "auc", "auc_low", "auc_high")
for s, step in (("2a", "2"), ("2b", "2"), ("3a", "3"), ("3b", "3")):
    if FOUND[s] is None: continue
    for name in (("grid_table.csv",) if s.endswith("a") else ("probe_table.csv", "tsfm_table.csv")):
        t = mf.read_csv(FOUND[s] / name)
        if t is None: continue
        t = t[t.cell == HEADLINE[step]]
        if s.endswith("b"):
            t = t[~t.model.str.contains("scratch|HGB|prior", regex=True)]
            if name == "tsfm_table.csv": t = t[t.raw.str.startswith("tsfm_")]
        add(step, t, "model", "roc_auc", "auc_ci_low", "auc_ci_high")
if FOUND["4a"] is not None and (FOUND["4a"] / "ladder.csv").exists():
    t = pd.read_csv(FOUND["4a"] / "ladder.csv"); t = t[t.cell == HEADLINE["4"]]
    for _, r in t.iterrows():
        rows.append(dict(step="4", label=r.rung, family="tree" if "HGB" in r.rung else "features" if "numbers only" in r.rung
                         else "image", auc=r.auc, low=r.low, high=r.high))
SUMMARY = pd.DataFrame(rows).drop_duplicates(["step", "label"])
SUMMARY.to_csv(OUT / "summary.csv", index=False)
display(SUMMARY.round(4))
if len(SUMMARY):
    mf.mindflow_figure(SUMMARY, OUT / "mindflow.png")

## 4. Parameter counts of every model (advisor rule)
Networks: weights; HGB: tree nodes (largest cell); probes: trainable head (encoder frozen, listed separately);
foundation models: all weights (frozen, zero-shot).

In [ ]:
prow = []
for s, p in FOUND.items():
    prow += mf.parameter_rows(s, p)
PARAMS = mf.parameter_table(prow)
PARAMS.to_csv(OUT / "parameters.csv", index=False)
display(PARAMS)

## 5. Feature selection (Step 4b)

In [ ]:
if FOUND["4b"] is not None:
    for f in sorted(FOUND["4b"].glob("*/confirmation.csv")):
        print(f.parent.name); display(pd.read_csv(f).round(4))